# Bulk LISS-IV pre/post scene preparation (2026)

This notebook uses the same functions as the browser GUI. It recursively reads NRSC product folders, extracts Path, Row, and DateOfPass from BAND_META.txt or .meta, groups BAND2.tif, BAND3.tif, and BAND4.tif, pairs scenes in the Jan–May and Oct–Dec 2026 windows, and writes each pair to a common CRS, grid, and shared overlap extent.

**Important:** aligning map grids does not estimate or remove residual image-to-image shifts. Confirm co-registration before using the outputs for PCA or landslide interpretation. If a product lacks georeferencing and does not include a supported GEOREF map extent, the notebook reports it for manual georeferencing; path/row alone cannot create coordinates.


In [ ]:
from pathlib import Path
import pandas as pd
import bulk_scene_prep as bulk

# Change these three local Windows paths.
PRE_ROOT = Path(r"F:\SILAS\Pre_Monsoon_2026")
POST_ROOT = Path(r"F:\SILAS\Post_Monsoon_2026")
OUTPUT_ROOT = Path(r"F:\SILAS\Aligned_Pairs_2026")

# The best spatial-coverage pair per path/row is recommended. Other
# candidate dates remain visible for manual review.
# A pair is auto-selected only when both scenes cover at least this share
# of their own footprint. Lower-scoring pairs remain visible for review.
MIN_COVERAGE = 0.80
RESAMPLING = "bilinear"  # imagery: bilinear, near, or cubic
TARGET_CRS = None        # None uses each pair's pre-scene CRS, e.g. "EPSG:32643"
TARGET_RESOLUTION = None # None uses the pre-scene pixel size in target CRS units
PROCESS_LOW_OVERLAP = False


In [ ]:
pre_scenes = bulk.scan_root(PRE_ROOT, ignore_root=OUTPUT_ROOT)
post_scenes = bulk.scan_root(POST_ROOT, ignore_root=OUTPUT_ROOT)

# De-duplicate in case the two selected roots overlap.
scenes_by_folder = {}
for scene in pre_scenes + post_scenes:
    scenes_by_folder[(scene["folder"], scene.get("date"))] = scene
scenes = list(scenes_by_folder.values())

inventory = pd.DataFrame(bulk.scene_report_rows(scenes))
display(inventory)
print(f"Found {len(scenes)} product folders; pre={sum(s['season'] == 'pre' for s in scenes)}, post={sum(s['season'] == 'post' for s in scenes)}")


In [ ]:
pairs = bulk.pair_scenes(scenes, min_coverage=MIN_COVERAGE)
pair_table = pd.DataFrame(bulk.pair_report_rows(pairs))
display(pair_table)
print(f"Found {len(pairs)} same-path/row pre/post candidates.")


In [ ]:
# Only the recommended READY pair per path/row is processed by default.
# Set PROCESS_LOW_OVERLAP=True after reviewing the coverage columns above.
allowed = {"READY", "LOW_OVERLAP_REVIEW"} if PROCESS_LOW_OVERLAP else {"READY"}
selected_pairs = [pair for pair in pairs if pair["status"] in allowed and pair.get("recommended")]
scene_index = {scene["folder"]: scene for scene in scenes}
print(f"Selected {len(selected_pairs)} pairs for output.")


In [ ]:
results = []
for pair in selected_pairs:
    pre = scene_index[pair["pre_folder"]]
    post = scene_index[pair["post_folder"]]
    try:
        result = bulk.prepare_pair(
            pre,
            post,
            OUTPUT_ROOT,
            target_crs=TARGET_CRS,
            target_resolution=TARGET_RESOLUTION,
            resampling=RESAMPLING,
            overwrite=False,
        )
        result.update({"status": "SUCCESS", "path": pair["path"], "row": pair["row"]})
    except Exception as exc:
        result = {
            "status": "FAILED",
            "path": pair["path"],
            "row": pair["row"],
            "pre_date": pair["pre_date"],
            "post_date": pair["post_date"],
            "error": str(exc),
        }
    results.append(result)

display(pd.DataFrame(results))


## Output layout

Each successful pair is placed under OUTPUT_ROOT/Path_####_Row_####/. The pair folder contains Pre_YYYYMMDD.tif, Post_YYYYMMDD.tif, and pair_manifest.json. Both rasters have identical dimensions, CRS, geotransform, pixel size, and the geographic overlap is cropped to one common output extent. The inventory and pair tables can be saved from their displayed DataFrames.

If a pair is marked NEEDS_GEOREFERENCE, review the source product type and its RPC/GCP metadata. The code can rebuild a missing transform only for an NRSC GEOREF product whose metadata includes UTM zone and product map bounds. It does not fabricate a georeference or claim that matching grids prove sub-pixel co-registration.
